Applying Tversky transformations to environmental audio

In [3]:
# imports

import torch
from torch.utils.data import DataLoader
import torchvision.transforms as tt
import torchvision.models as models
import math

import torch.nn as nn

import torch.nn.functional as F

from pyha_analyzer.preprocessors import MelSpectrogramPreprocessors
from tqdm.notebook import tqdm

import numpy as np
import matplotlib.pyplot as plt

In [4]:
config = {
    "learning_rate": 2e-3,
    "learning_rate_decay": 0,
    "device": 'cuda',
    "seed": 1
}

In [5]:
torch.manual_seed(config["seed"])

In [6]:
from datasets import load_dataset

mnist_dataset = load_dataset("mnist")

train = mnist_dataset["train"]
test = mnist_dataset["test"]

def transform(batch):
  t = tt.Compose([
    tt.Grayscale(num_output_channels=3),
    tt.PILToTensor(),
    tt.ConvertImageDtype(torch.float)
    ])
  
  batch['image'] = [t(img) for img in batch['image']]
  batch['label'] = F.one_hot(torch.tensor(batch['label']), num_classes=10)
  
  return batch

train.set_transform(transform)
test.set_transform(transform)

In [7]:
class Tversky(nn.Module):
    """
    Similar to a fully-connected layer, but computes Tversky similarity instead
    """
    def __init__(
        self,
        in_features: tuple,
        out_features: int,
        num_features=256,
        feature_bank=None,
        phi=torch.mul,
        substract=True,
        device=None,
        dtype=None
    ) -> None:
        factory_kwargs = {"device": device, "dtype": dtype}
        super().__init__()
        
        self.phi = phi
        self.substract = substract
        
        if feature_bank is None:
            self.feature_bank = nn.Parameter(
                torch.empty((num_features,) + in_features, **factory_kwargs)
            )
        else:
            self.feature_bank = feature_bank
            
        self.prototypes = nn.Parameter(
                torch.empty( (out_features,) + in_features, **factory_kwargs)
            )
            
        self.alpha = nn.Parameter(torch.empty(1))
        self.beta = nn.Parameter(torch.empty(1))
        self.theta = nn.Parameter(torch.empty(1))

        self.reset_parameters()
        
    def reset_parameters(self) -> None:
        nn.init.uniform_(self.feature_bank)
        nn.init.uniform_(self.prototypes)
        nn.init.uniform_(self.alpha)
        nn.init.uniform_(self.beta)
        nn.init.uniform_(self.theta)
    
    def forward(self, input: torch.Tensor):
        
        a_f = (input.flatten(1, -1).unsqueeze(1) * self.feature_bank.flatten(1, -1).unsqueeze(0)).sum(-1).unsqueeze(1)
        p_f = (self.prototypes.flatten(1, -1).unsqueeze(1) * self.feature_bank.flatten(1, -1).unsqueeze(0)).sum(-1).unsqueeze(0)
        

        val = self.phi(a_f, p_f)
        mask = (torch.minimum(a_f, p_f) >= 0).int()
        intersection = self.theta * (val * mask).sum(-1)
        
        
        if self.substract:
            alpha_difference = -self.alpha * (a_f * ((a_f > 0) | (p_f <= 0)).int()).sum(-1)
            beta_difference = -self.beta * (p_f * ((p_f > 0) | (a_f <= 0)).int()).sum(-1)
        else:
            alpha_difference = -self.alpha * ((a_f - p_f) * ((a_f > 0) | (p_f > 0) | (a_f > p_f)).int()).sum(-1)
            beta_difference = -self.beta * ((p_f - a_f) * ((p_f > 0) | (a_f > 0) | (p_f > a_f)).int()).sum(-1)
        
        return intersection + alpha_difference + beta_difference
    
t = Tversky((4,), 10, 5)

A = torch.rand((2, 4))

t(A).shape

torch.Size([2, 10])

In [8]:
ResNet = models.resnet50(weights=None)
ResNet.fc = nn.Sequential(
    Tversky((ResNet.fc.in_features,),10, 20, substract=True)
)

# ResNet = models.resnet50(weights=None)
# ResNet.fc = nn.Linear(ResNet.fc.in_features, 10)

model = nn.Sequential(
    nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False),
    ResNet
).to(config['device'])

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), weight_decay=0, lr=config["learning_rate"], betas=(0.8, 0.999))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda epoch : (1-config["learning_rate_decay"])**epoch)
metric = torch.nn.CrossEntropyLoss()
train_loader = DataLoader(train, batch_size=8, num_workers=8, shuffle=True)

In [8]:
step, loss = 0, []

correct, total = 0, 0

for epoch in range(5):
    for data in tqdm(train_loader, desc=str(epoch), leave=False):
        img = data['image']

        optimizer.zero_grad()
        x = img.to(config["device"])
        
        pred = model(x)
        
        label = data['label'].to(config["device"], torch.float)
        
        score = metric(pred, label)
        score.backward()
        optimizer.step()
        
        loss.append(score.item())
        
        
        if step % 500 == 0:
            with torch.no_grad():
                print(np.mean(loss[-10:-1]))
                
                print(pred.argmax(dim=-1))
                print(label.argmax(dim=-1))
            
        step += 1

0:   0%|          | 0/7500 [00:00<?, ?it/s]

/home/a.jajodia.229/acoustic/anu_experiments_acoustic_species/.venv/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/a.jajodia.229/acoustic/anu_experiments_acoustic_species/.venv/lib/python3.11/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


nan
tensor([9, 9, 9, 9, 9, 9, 9, 9], device='cuda:0')
tensor([9, 9, 9, 1, 6, 9, 8, 7], device='cuda:0')
7.361424816979302
tensor([2, 7, 4, 7, 2, 4, 7, 7], device='cuda:0')
tensor([2, 4, 7, 1, 2, 6, 5, 1], device='cuda:0')
11.68199790848626
tensor([1, 1, 1, 1, 1, 6, 1, 6], device='cuda:0')
tensor([9, 4, 3, 7, 4, 1, 2, 2], device='cuda:0')
2.3220419353908963
tensor([4, 0, 0, 2, 5, 8, 8, 5], device='cuda:0')
tensor([0, 0, 1, 4, 5, 1, 2, 5], device='cuda:0')
2.324885288874308
tensor([6, 0, 8, 8, 8, 9, 8, 8], device='cuda:0')
tensor([2, 6, 4, 4, 5, 1, 6, 2], device='cuda:0')
2.5428415404425726
tensor([3, 5, 5, 0, 5, 5, 0, 0], device='cuda:0')
tensor([6, 3, 7, 5, 6, 2, 5, 2], device='cuda:0')
80.2886175579495
tensor([2, 0, 2, 0, 0, 0, 2, 2], device='cuda:0')
tensor([0, 7, 8, 8, 9, 2, 2, 5], device='cuda:0')
2.6356181568569608
tensor([4, 4, 4, 1, 6, 1, 4, 4], device='cuda:0')
tensor([9, 3, 4, 8, 3, 5, 3, 2], device='cuda:0')
2.8871326446533203
tensor([0, 0, 0, 8, 8, 0, 8, 8], device='cuda:0')

1:   0%|          | 0/7500 [00:00<?, ?it/s]

1.89709468682607
tensor([3, 0, 3, 8, 3, 8, 3, 1], device='cuda:0')
tensor([5, 0, 4, 7, 3, 6, 5, 1], device='cuda:0')
1.72466954920027
tensor([1, 9, 9, 0, 0, 0, 0, 0], device='cuda:0')
tensor([1, 6, 4, 6, 6, 6, 6, 6], device='cuda:0')
1.5834727552202013
tensor([9, 8, 8, 3, 4, 3, 1, 9], device='cuda:0')
tensor([6, 3, 9, 9, 1, 3, 1, 8], device='cuda:0')
1.829568041695489
tensor([5, 5, 5, 5, 1, 5, 7, 8], device='cuda:0')
tensor([3, 0, 2, 2, 1, 5, 6, 8], device='cuda:0')
1.4824602339002821
tensor([7, 9, 8, 6, 3, 7, 8, 7], device='cuda:0')
tensor([7, 0, 7, 6, 5, 6, 8, 0], device='cuda:0')
6.891280108027988
tensor([4, 2, 9, 2, 0, 2, 2, 4], device='cuda:0')
tensor([7, 5, 5, 9, 3, 3, 5, 2], device='cuda:0')
3.7360040876600475
tensor([3, 2, 3, 3, 3, 2, 0, 0], device='cuda:0')
tensor([1, 2, 4, 3, 3, 2, 0, 7], device='cuda:0')
4.741213533613417
tensor([3, 1, 1, 1, 6, 1, 2, 1], device='cuda:0')
tensor([8, 6, 3, 2, 2, 1, 0, 5], device='cuda:0')
2.845252831776937
tensor([1, 4, 1, 1, 1, 7, 1, 6], devi

...

1.807713336414761
tensor([9, 2, 1, 6, 2, 3, 6, 4], device='cuda:0')
tensor([9, 3, 4, 3, 5, 9, 6, 4], device='cuda:0')
1.9133744769626193
tensor([2, 8, 7, 2, 5, 7, 0, 0], device='cuda:0')
tensor([6, 8, 8, 2, 2, 9, 6, 0], device='cuda:0')
1.2898574206564162
tensor([5, 9, 8, 2, 5, 6, 8, 4], device='cuda:0')
tensor([5, 9, 8, 2, 5, 6, 8, 9], device='cuda:0')
1.027351717154185
tensor([0, 7, 7, 9, 5, 9, 1, 3], device='cuda:0')
tensor([6, 5, 3, 4, 5, 4, 1, 2], device='cuda:0')
1.1240343186590407
tensor([8, 3, 4, 1, 1, 9, 5, 1], device='cuda:0')
tensor([0, 2, 4, 7, 7, 9, 3, 1], device='cuda:0')
1.0052445729573567
tensor([5, 0, 1, 1, 1, 8, 2, 0], device='cuda:0')
tensor([3, 0, 1, 1, 1, 0, 1, 2], device='cuda:0')
0.8345522483189901
tensor([0, 7, 0, 2, 2, 8, 4, 5], device='cuda:0')
tensor([0, 7, 0, 3, 2, 3, 0, 5], device='cuda:0')
0.4247028347518709
tensor([7, 1, 2, 0, 2, 4, 7, 6], device='cuda:0')
tensor([1, 1, 3, 0, 2, 6, 7, 6], device='cuda:0')
0.38531240158610874
tensor([0, 2, 7, 1, 6, 9, 2, 7

3:   0%|          | 0/7500 [00:00<?, ?it/s]

0.2250007697277599
tensor([1, 0, 2, 9, 6, 2, 8, 7], device='cuda:0')
tensor([1, 0, 0, 9, 6, 2, 8, 7], device='cuda:0')
0.5906284044419104
tensor([8, 1, 7, 0, 3, 5, 9, 0], device='cuda:0')
tensor([8, 1, 7, 0, 3, 9, 9, 0], device='cuda:0')
0.5629090687466992
tensor([1, 9, 5, 8, 1, 0, 2, 8], device='cuda:0')
tensor([1, 9, 5, 8, 1, 0, 2, 8], device='cuda:0')
0.395764800409476
tensor([5, 1, 7, 0, 7, 0, 3, 4], device='cuda:0')
tensor([5, 1, 7, 0, 7, 0, 3, 4], device='cuda:0')
0.1305489114796122
tensor([8, 0, 4, 0, 7, 9, 5, 0], device='cuda:0')
tensor([8, 0, 4, 0, 7, 9, 5, 0], device='cuda:0')
0.08100945455953479
tensor([3, 9, 7, 8, 7, 3, 7, 8], device='cuda:0')
tensor([3, 9, 7, 8, 7, 3, 7, 8], device='cuda:0')
0.1455152373140057
tensor([5, 8, 1, 7, 4, 2, 1, 2], device='cuda:0')
tensor([5, 8, 1, 7, 6, 2, 1, 2], device='cuda:0')
0.9436471851335632
tensor([0, 4, 3, 0, 4, 5, 4, 3], device='cuda:0')
tensor([0, 9, 7, 0, 9, 5, 4, 3], device='cuda:0')
0.15407996252179146
tensor([6, 4, 7, 9, 0, 7, 1,

4:   0%|          | 0/7500 [00:00<?, ?it/s]

0.23943747559355366
tensor([6, 7, 9, 2, 9, 0, 7, 7], device='cuda:0')
tensor([6, 7, 7, 2, 9, 0, 7, 7], device='cuda:0')
0.01727004602758421
tensor([6, 1, 2, 2, 7, 3, 9, 9], device='cuda:0')
tensor([6, 1, 2, 2, 7, 3, 9, 9], device='cuda:0')
0.06734813057765779
tensor([1, 7, 8, 3, 8, 7, 0, 4], device='cuda:0')
tensor([1, 7, 8, 3, 8, 7, 0, 4], device='cuda:0')
0.12376362033602265
tensor([1, 0, 3, 5, 4, 3, 0, 3], device='cuda:0')
tensor([1, 0, 3, 5, 4, 3, 0, 3], device='cuda:0')
0.022384443682514958
tensor([4, 7, 0, 2, 3, 0, 1, 9], device='cuda:0')
tensor([4, 7, 0, 2, 3, 0, 1, 9], device='cuda:0')
0.1380088504196869
tensor([7, 8, 2, 7, 6, 9, 2, 2], device='cuda:0')
tensor([7, 8, 2, 7, 6, 9, 2, 2], device='cuda:0')
0.15355824840824223
tensor([9, 3, 8, 7, 1, 0, 0, 7], device='cuda:0')
tensor([9, 3, 8, 3, 1, 2, 0, 7], device='cuda:0')
0.02545078105241474
tensor([7, 8, 6, 4, 0, 7, 8, 7], device='cuda:0')
tensor([7, 8, 6, 4, 0, 7, 8, 7], device='cuda:0')
0.048014780545296766
tensor([1, 7, 0, 3,

In [9]:
torch.save(model.state_dict(), "tversky_model.pt")

In [21]:
model.load_state_dict(torch.load("tversky_model.pt"))
model.eval().to(config["device"])

Sequential(
  (0): Upsample(size=(224, 224), mode='bilinear')
  (1): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=Tru

In [22]:
test_loader = DataLoader(test, batch_size=8, num_workers=8, shuffle=True)

accuracy = []

for data in tqdm(test_loader, leave=False):
    img = data['image']

    x = img.to(config["device"])
    
    pred = model(x)
    
    label = data['label'].to(config["device"], torch.float)
    
    accuracy.append((pred.argmax(dim=-1).detach() == label.argmax(dim=-1).detach()).float().mean(dim=-1))

  0%|          | 0/1250 [00:00<?, ?it/s]

In [29]:
torch.load("tversky_model.pt")['1.fc.0.prototypes'].shape

torch.Size([10, 2048])

In [23]:
print(torch.mean(torch.tensor(accuracy)))

tensor(0.9672)


In [30]:
ResNet = models.resnet50(weights=None)
ResNet.fc = nn.Linear(ResNet.fc.in_features, 10)

model = nn.Sequential(
    nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False),
    ResNet
).to(config['device'])

In [31]:
optimizer = torch.optim.AdamW(model.parameters(), weight_decay=0, lr=config["learning_rate"], betas=(0.8, 0.999))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda epoch : (1-config["learning_rate_decay"])**epoch)
metric = torch.nn.CrossEntropyLoss()
train_loader = DataLoader(train, batch_size=8, num_workers=8, shuffle=True)

In [32]:
step, loss = 0, []

correct, total = 0, 0

for epoch in range(5):
    for data in tqdm(train_loader, desc=str(epoch), leave=False):
        img = data['image']

        optimizer.zero_grad()
        x = img.to(config["device"])
        
        pred = model(x)
        
        label = data['label'].to(config["device"], torch.float)
        
        score = metric(pred, label)
        score.backward()
        optimizer.step()
        
        loss.append(score.item())
        
        
        if step % 500 == 0:
            with torch.no_grad():
                print(np.mean(loss[-10:-1]))
                
                print(pred.argmax(dim=-1))
                print(label.argmax(dim=-1))
            
        step += 1

0:   0%|          | 0/7500 [00:00<?, ?it/s]

/home/a.jajodia.229/acoustic/anu_experiments_acoustic_species/.venv/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/a.jajodia.229/acoustic/anu_experiments_acoustic_species/.venv/lib/python3.11/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


nan
tensor([7, 7, 7, 7, 7, 8, 7, 7], device='cuda:0')
tensor([9, 7, 5, 7, 2, 7, 0, 5], device='cuda:0')
0.5166097610361047
tensor([2, 7, 8, 0, 4, 7, 3, 9], device='cuda:0')
tensor([2, 7, 8, 5, 4, 7, 3, 9], device='cuda:0')
0.5509804868035846
tensor([1, 3, 8, 2, 8, 6, 2, 0], device='cuda:0')
tensor([1, 3, 8, 2, 0, 6, 2, 0], device='cuda:0')
0.1563916768775218
tensor([6, 3, 8, 7, 4, 9, 2, 5], device='cuda:0')
tensor([6, 5, 8, 7, 4, 9, 2, 5], device='cuda:0')
0.3645916272782617
tensor([4, 3, 1, 2, 0, 6, 6, 0], device='cuda:0')
tensor([4, 8, 1, 2, 0, 6, 6, 0], device='cuda:0')
0.1442848803061578
tensor([7, 5, 9, 8, 2, 0, 9, 2], device='cuda:0')
tensor([7, 5, 9, 8, 7, 0, 9, 2], device='cuda:0')
0.35701277800318265
tensor([6, 4, 8, 2, 3, 2, 8, 3], device='cuda:0')
tensor([6, 4, 8, 2, 3, 2, 8, 3], device='cuda:0')
0.23289155370245376
tensor([3, 4, 9, 7, 5, 7, 6, 1], device='cuda:0')
tensor([3, 4, 9, 7, 5, 7, 6, 1], device='cuda:0')
0.22683924860838386
tensor([8, 5, 0, 4, 8, 8, 1, 0], device='

1:   0%|          | 0/7500 [00:00<?, ?it/s]

0.16550025266284743
tensor([5, 1, 3, 9, 5, 4, 5, 4], device='cuda:0')
tensor([5, 1, 3, 9, 5, 4, 5, 4], device='cuda:0')
0.04834876349195838
tensor([6, 7, 6, 8, 3, 7, 3, 3], device='cuda:0')
tensor([6, 7, 6, 8, 3, 7, 3, 3], device='cuda:0')
0.12795747262943122
tensor([8, 0, 5, 9, 7, 9, 4, 4], device='cuda:0')
tensor([8, 6, 5, 9, 7, 9, 4, 4], device='cuda:0')
0.008916699275788333
tensor([7, 8, 2, 8, 4, 0, 6, 1], device='cuda:0')
tensor([7, 8, 2, 8, 4, 0, 6, 1], device='cuda:0')
0.12640297690975583
tensor([9, 2, 7, 1, 0, 3, 1, 1], device='cuda:0')
tensor([9, 2, 7, 1, 0, 3, 1, 1], device='cuda:0')
0.05966495494875643
tensor([1, 1, 2, 9, 8, 2, 1, 1], device='cuda:0')
tensor([1, 1, 2, 9, 8, 2, 1, 1], device='cuda:0')
0.012943122419528663
tensor([3, 0, 0, 4, 3, 1, 8, 6], device='cuda:0')
tensor([3, 0, 0, 4, 3, 1, 8, 6], device='cuda:0')
0.37716660860718954
tensor([1, 4, 4, 5, 8, 2, 0, 3], device='cuda:0')
tensor([1, 4, 4, 5, 8, 2, 0, 3], device='cuda:0')
0.27102141700581545
tensor([7, 1, 0, 4

...

0.015312574221752584
tensor([8, 3, 4, 6, 0, 6, 6, 5], device='cuda:0')
tensor([8, 3, 4, 6, 0, 6, 6, 5], device='cuda:0')
0.008741037949221209
tensor([5, 5, 4, 5, 7, 2, 0, 0], device='cuda:0')
tensor([5, 5, 4, 5, 7, 2, 0, 0], device='cuda:0')
0.0362441571843293
tensor([1, 5, 8, 2, 5, 3, 0, 6], device='cuda:0')
tensor([1, 5, 8, 2, 5, 3, 0, 6], device='cuda:0')
0.0891960131444244
tensor([6, 9, 6, 4, 4, 9, 3, 2], device='cuda:0')
tensor([6, 9, 6, 4, 4, 9, 8, 2], device='cuda:0')
0.08537025703764003
tensor([9, 8, 0, 8, 0, 4, 1, 5], device='cuda:0')
tensor([9, 8, 0, 8, 0, 4, 1, 5], device='cuda:0')
0.048687385131617904
tensor([1, 0, 4, 4, 7, 7, 6, 2], device='cuda:0')
tensor([1, 0, 4, 4, 7, 7, 6, 2], device='cuda:0')
0.08251399263746054
tensor([3, 7, 3, 0, 7, 6, 8, 1], device='cuda:0')
tensor([3, 7, 3, 0, 7, 6, 8, 1], device='cuda:0')
0.15599997195749893
tensor([7, 0, 1, 9, 9, 7, 9, 7], device='cuda:0')
tensor([7, 0, 1, 9, 9, 7, 9, 7], device='cuda:0')
0.004631123849119629
tensor([1, 4, 0, 7

3:   0%|          | 0/7500 [00:00<?, ?it/s]

0.003598576971045178
tensor([9, 4, 3, 0, 0, 5, 3, 1], device='cuda:0')
tensor([9, 4, 3, 0, 0, 5, 3, 1], device='cuda:0')
0.034922311147965956
tensor([2, 7, 6, 1, 6, 8, 3, 4], device='cuda:0')
tensor([2, 7, 6, 1, 6, 8, 3, 4], device='cuda:0')
0.01048201645931436
tensor([1, 3, 8, 0, 0, 6, 7, 6], device='cuda:0')
tensor([1, 3, 8, 0, 0, 6, 7, 6], device='cuda:0')
0.019573158267626747
tensor([4, 4, 5, 7, 2, 4, 6, 2], device='cuda:0')
tensor([4, 4, 5, 7, 2, 4, 6, 2], device='cuda:0')
0.003770792744944629
tensor([6, 0, 6, 6, 8, 8, 3, 2], device='cuda:0')
tensor([6, 0, 6, 6, 8, 8, 3, 2], device='cuda:0')
0.01656741321979401
tensor([0, 3, 7, 7, 2, 0, 8, 1], device='cuda:0')
tensor([0, 3, 7, 7, 2, 0, 8, 1], device='cuda:0')
0.025993393105131365
tensor([3, 7, 1, 8, 6, 8, 6, 3], device='cuda:0')
tensor([3, 7, 1, 8, 6, 8, 6, 3], device='cuda:0')
0.024374499256535072
tensor([1, 1, 6, 5, 3, 7, 5, 3], device='cuda:0')
tensor([1, 1, 6, 5, 3, 7, 5, 3], device='cuda:0')
0.01197355959589509
tensor([2, 6, 

4:   0%|          | 0/7500 [00:00<?, ?it/s]

0.12235117170105998
tensor([7, 2, 4, 8, 8, 3, 4, 1], device='cuda:0')
tensor([7, 2, 4, 8, 8, 3, 4, 1], device='cuda:0')
0.004164256310711305
tensor([2, 0, 6, 2, 3, 3, 2, 9], device='cuda:0')
tensor([2, 0, 6, 2, 3, 3, 2, 9], device='cuda:0')
0.02054821694410849
tensor([9, 3, 3, 5, 1, 2, 4, 0], device='cuda:0')
tensor([9, 3, 3, 5, 1, 2, 4, 0], device='cuda:0')
0.13624465937735092
tensor([4, 9, 8, 0, 3, 2, 5, 1], device='cuda:0')
tensor([4, 9, 8, 0, 3, 2, 5, 1], device='cuda:0')
0.07735581783830033
tensor([4, 9, 2, 4, 0, 2, 1, 7], device='cuda:0')
tensor([4, 9, 2, 4, 0, 2, 1, 7], device='cuda:0')
0.035485013551200005
tensor([7, 0, 3, 3, 4, 8, 4, 4], device='cuda:0')
tensor([7, 0, 3, 3, 4, 8, 4, 4], device='cuda:0')
0.003951255162772011
tensor([3, 8, 2, 8, 1, 7, 6, 2], device='cuda:0')
tensor([3, 8, 2, 8, 1, 7, 6, 2], device='cuda:0')
0.10981897033505245
tensor([7, 1, 5, 9, 6, 0, 1, 0], device='cuda:0')
tensor([7, 1, 5, 9, 6, 0, 1, 0], device='cuda:0')
0.014373586409621768
tensor([5, 6, 2,

In [33]:
test_loader = DataLoader(test, batch_size=8, num_workers=8, shuffle=True)

accuracy = []

for data in tqdm(test_loader, leave=False):
    img = data['image']

    x = img.to(config["device"])
    
    pred = model(x)
    
    label = data['label'].to(config["device"], torch.float)
    
    accuracy.append((pred.argmax(dim=-1).detach() == label.argmax(dim=-1).detach()).float().mean(dim=-1))

  0%|          | 0/1250 [00:00<?, ?it/s]

In [34]:
print(torch.mean(torch.tensor(accuracy)))

tensor(0.9907)
